# Flatten instruments
Project normalized instrument lifecycle rows from the sorted market-log stream.

In [ ]:
source = "logs.market"
target = "market.instruments"
start = None
end = None
catalog = "rekep"
catalog_properties = {}
merge_by = True
commit_row_size = 250_000

In [ ]:
import datetime

import pyarrow
from pyiceberg.expressions import And, EqualTo, GreaterThanOrEqual, LessThan

from rekep.iceberg import IcebergDataset
from rekep.market import EventType, Instrument
from rekep.text import Log


def _unix_ns(value, *, upper=False):
    if value is None:
        return None
    text = str(value)
    date_only = len(text) == 10
    instant = datetime.datetime.fromisoformat(text.replace("Z", "+00:00"))
    if instant.tzinfo is None:
        instant = instant.replace(tzinfo=datetime.UTC)
    instant = instant.astimezone(datetime.UTC)
    if upper and date_only:
        instant += datetime.timedelta(days=1)
    epoch = datetime.datetime(1970, 1, 1, tzinfo=datetime.UTC)
    return (instant - epoch) // datetime.timedelta(microseconds=1) * 1_000


lower, upper = _unix_ns(start), _unix_ns(end, upper=True)


def _window(column="unix"):
    predicates = [
        EqualTo("etype", int(EventType.INSTRUMENT)),
        EqualTo("driver_name", Log.into_instrument_driver()),
    ]
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return predicates[0] if len(predicates) == 1 else And(*predicates)


logs_table = IcebergDataset(name=source, catalog=catalog, properties=dict(catalog_properties))
instrument_table = IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    struct=Instrument.into_field(),
    commit_row_size=commit_row_size,
    sort_by=("unix", "version", "hash"),
)


def _logs():
    reader = logs_table.read_arrow_reader(
        Log.into_field(), row_filter=_window(), order_by=("unix", "seq", "hash")
    )
    for batch in reader:
        for row in batch.to_pylist():
            yield Log.from_dict(row)


def _flush(held):
    if not held:
        return 0
    table = pyarrow.Table.from_pylist(
        [instrument.into_dict() for instrument in held],
        schema=Instrument.into_field().into_arrow_schema(),
    )
    held.clear()
    return instrument_table.append_arrow_table(table, merge_by=merge_by)


held = []
read = written = 0
for log in _logs():
    instrument = log.into_instrument()
    if instrument is None:
        continue
    held.append(instrument)
    read += 1
    if len(held) >= commit_row_size:
        written += _flush(held)
written += _flush(held)
result = {"versions": read, "written": written, "target": target}
result